<a href="https://colab.research.google.com/github/turinayojonas/NSAI/blob/main/Final_dataset_and_excersices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

N = 50000  # observations

# ---------- Reference lists ----------
provinces_districts = {
    "Kigali City": ["Nyarugenge", "Gasabo", "Kicukiro"],
    "Southern Province": ["Huye", "Muhanga", "Nyanza", "Gisagara", "Nyaruguru", "Kamonyi", "Ruhango", "Nyamagabe"],
    "Northern Province": ["Musanze", "Gicumbi", "Rulindo", "Burera", "Gakenke"],
    "Eastern Province": ["Rwamagana", "Kayonza", "Kirehe", "Ngoma", "Nyagatare", "Gatsibo", "Bugesera"],
    "Western Province": ["Rubavu", "Rusizi", "Karongi", "Nyamasheke", "Ngororero", "Rutsiro", "Nyabihu"],
}
provinces = list(provinces_districts.keys())

branches = [
    ("BR001", "Kigali Main Branch"), ("BR002", "Nyarugenge Branch"), ("BR003", "Remera Branch"),
    ("BR004", "Kimironko Branch"), ("BR005", "Huye Branch"), ("BR006", "Musanze Branch"),
    ("BR007", "Rubavu Branch"), ("BR008", "Rwamagana Branch"), ("BR009", "Muhanga Branch"),
    ("BR010", "Nyagatare Branch"), ("BR011", "Karongi Branch"), ("BR012", "Gicumbi Branch"),
    ("BR013", "Kayonza Branch"), ("BR014", "Rusizi Branch"), ("BR015", "Nyanza Branch"),
]

sectors = ["Agriculture", "Manufacturing", "Trade & Commerce", "Construction & Real Estate",
           "Transport & Logistics", "Mining & Quarrying", "Energy & Utilities", "ICT",
           "Education", "Health", "Hospitality & Tourism", "Financial Services"]

product_types = ["Mortgage Loan", "Personal Loan", "Auto Loan", "Business Term Loan",
                 "Agriculture Loan", "Overdraft Facility", "Trade Finance", "Microfinance Loan",
                 "Student Loan", "Salary Advance"]

customer_types = ["Individual", "SME", "Corporate", "Microfinance Client", "Public Institution"]

collateral_types = ["Land Title", "Residential Property", "Commercial Property", "Vehicle",
                     "Fixed Deposit", "Equipment", "Inventory Pledge", "None (Unsecured)", "Guarantee"]

classifications = ["Normal", "Watch", "Substandard", "Doubtful", "Loss"]
prov_rate_by_class = {"Normal": 1.0, "Watch": 3.0, "Substandard": 20.0, "Doubtful": 50.0, "Loss": 100.0}
dpd_range_by_class = {"Normal": (0, 29), "Watch": (30, 89), "Substandard": (90, 179),
                       "Doubtful": (180, 359), "Loss": (360, 900)}

officers_first = ["Eric", "Claudine", "Jean Paul", "Aline", "Emmanuel", "Solange", "Patrick", "Diane",
                   "Innocent", "Chantal", "Olivier", "Yvonne", "Vincent", "Beatrice", "Fabrice", "Marie Claire"]
officers_last = ["Uwase", "Habimana", "Mugisha", "Niyonsenga", "Ingabire", "Bizimana", "Umutoni",
                  "Nsengiyumva", "Mukamana", "Rugamba", "Kalisa", "Uwimana"]
loan_officers = [f"{f} {l}" for f in officers_first for l in officers_last]
random.shuffle(loan_officers)
loan_officers = loan_officers[:60]

first_names_m = ["Jean", "Eric", "Patrick", "Emmanuel", "Innocent", "Vincent", "Olivier", "Fabrice",
                  "Theogene", "Celestin", "Damien", "Gilbert", "Herve", "Ivan", "Justin", "Kevin"]
first_names_f = ["Claudine", "Aline", "Solange", "Diane", "Chantal", "Yvonne", "Beatrice", "Marie Claire",
                  "Grace", "Josiane", "Liliane", "Nadia", "Odette", "Pascasie", "Rachel", "Sandrine"]
last_names = ["Uwase", "Habimana", "Mugisha", "Niyonsenga", "Ingabire", "Bizimana", "Umutoni",
              "Nsengiyumva", "Mukamana", "Rugamba", "Kalisa", "Uwimana", "Mutesi", "Ndayisenga",
              "Uwizeyimana", "Twagirayezu", "Nkurunziza", "Gasana", "Mukeshimana", "Byiringiro"]

corp_words1 = ["Rwanda", "Kivu Lake", "Mille Collines", "Umuganda", "Virunga", "Akagera", "Nyandungu",
               "Ubumwe", "Ikaze", "Isange", "Amahoro", "Kigeli"]
corp_words2 = ["Trading Ltd", "Enterprises", "Ventures", "Holdings", "General Merchandise", "Agro Ltd",
               "Construction Co", "Logistics Ltd", "Group", "Investments"]

data_sources = ["CoreBanking-T24", "CoreBanking-Flexcube", "LoanOriginationSystem", "LegacyMigration2019"]
kyc_statuses = ["Complete", "Incomplete", "Pending Review", "Expired - Needs Update"]
employment = ["Salaried - Private", "Salaried - Public Sector", "Self-Employed", "Business Owner",
              "Farmer", "Unemployed", "Retired"]
repay_freq = ["Monthly", "Quarterly", "Bullet (End of Term)", "Seasonal (Agriculture)"]
currencies = ["RWF"] * 92 + ["USD"] * 6 + ["EUR"] * 2

def rand_date(start, end):
    if end <= start:
        return start
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))

start_hist = datetime(2019, 1, 1)
today = datetime(2026, 6, 30)

rows = []
customer_pool_size = int(N * 0.62)  # some customers have multiple loans
customer_records = {}
for i in range(customer_pool_size):
    cid = f"CUS{100000+i}"
    ctype = random.choices(customer_types, weights=[55, 25, 10, 6, 4])[0]
    if ctype in ("Individual", "Microfinance Client"):
        gender = random.choice(["Male", "Female"])
        name = f"{random.choice(first_names_m if gender=='Male' else first_names_f)} {random.choice(last_names)}"
    else:
        gender = "N/A"
        name = f"{random.choice(corp_words1)} {random.choice(corp_words2)}"
    customer_records[cid] = (name, ctype, gender)

customer_ids = list(customer_records.keys())

for i in range(N):
    loan_id = f"LN{2019000000+i}"
    cid = random.choices(customer_ids, weights=None, k=1)[0] if random.random() < 0.75 else random.choice(customer_ids)
    name, ctype, gender = customer_records[cid]

    province = random.choice(provinces)
    district = random.choice(provinces_districts[province])
    branch_code, branch_name = random.choice(branches)
    officer = random.choice(loan_officers)
    product = random.choices(product_types, weights=[8, 22, 10, 15, 10, 8, 6, 14, 4, 3])[0]
    sector = random.choice(sectors)
    currency = random.choice(currencies)

    age = int(np.clip(np.random.normal(39, 11), 19, 78)) if ctype in ("Individual", "Microfinance Client") else None

    app_date = rand_date(start_hist, today - timedelta(days=30))
    approval_date = app_date + timedelta(days=random.randint(1, 21))
    disb_date = approval_date + timedelta(days=random.randint(0, 14))
    term_months = random.choice([6, 12, 18, 24, 36, 48, 60, 84, 120, 180, 240])
    maturity_date = disb_date + timedelta(days=int(term_months * 30.4))

    if currency == "RWF":
        principal = float(np.random.choice(
            [random.uniform(200000, 3000000), random.uniform(3000000, 25000000),
             random.uniform(25000000, 500000000)], p=[0.55, 0.35, 0.10]
        ))
    elif currency == "USD":
        principal = random.uniform(2000, 400000)
    else:
        principal = random.uniform(1500, 300000)
    principal = round(principal, 2)

    rate = round(random.uniform(9.0, 22.5), 2)

    classification = random.choices(classifications, weights=[74, 12, 6, 4, 4])[0]
    dpd_lo, dpd_hi = dpd_range_by_class[classification]
    dpd = random.randint(dpd_lo, dpd_hi)
    missed = 0 if dpd == 0 else max(1, dpd // 30)

    pct_repaid = np.clip(np.random.beta(2, 2) if classification == "Normal" else np.random.beta(1.2, 3), 0.0, 0.98)
    amount_repaid = round(principal * pct_repaid, 2)
    outstanding_principal = round(principal - amount_repaid, 2)
    outstanding_interest = round(outstanding_principal * (rate / 100) * random.uniform(0.02, 0.25), 2)
    total_outstanding = round(outstanding_principal + outstanding_interest, 2)

    prov_rate = prov_rate_by_class[classification]
    provision_amount = round(total_outstanding * (prov_rate / 100), 2)

    freq = random.choice(repay_freq)
    n_installments = term_months

    collateral_needed = product not in ("Overdraft Facility", "Salary Advance", "Student Loan")
    collateral = random.choice(collateral_types) if collateral_needed else random.choice(["None (Unsecured)", "Guarantee"])
    if collateral == "None (Unsecured)":
        collateral_value = 0.0
    else:
        collateral_value = round(principal * random.uniform(0.8, 2.2), 2)
    ltv = round((principal / collateral_value * 100), 2) if collateral_value > 0 else None

    credit_score = int(np.clip(np.random.normal(640, 90), 300, 850))
    kyc = random.choices(kyc_statuses, weights=[78, 8, 9, 5])[0]
    restructured = random.choices(["Yes", "No"], weights=[7, 93])[0]
    write_off = "Yes" if classification == "Loss" and random.random() < 0.35 else "No"
    write_off_amount = round(total_outstanding * random.uniform(0.5, 1.0), 2) if write_off == "Yes" else 0.0
    last_review = rand_date(disb_date, today) if random.random() < 0.85 else None
    approving_authority = random.choice(["Branch Credit Committee", "Regional Credit Committee",
                                          "Head Office Credit Committee", "Board Credit Committee"])
    insurance = random.choices(["Yes", "No"], weights=[62, 38])[0]
    guarantor = f"{random.choice(first_names_m+first_names_f)} {random.choice(last_names)}" if random.random() < 0.4 else None
    emp_status = random.choice(employment) if ctype in ("Individual", "Microfinance Client") else "N/A"
    monthly_income = round(random.uniform(80000, 4500000), 0) if ctype in ("Individual", "Microfinance Client") else None
    dti = round(random.uniform(5, 75), 1) if monthly_income else None
    source_sys = random.choice(data_sources)

    rows.append([
        loan_id, cid, name, ctype, gender, age, sector, province, district, branch_code, branch_name,
        officer, product, currency, app_date, approval_date, disb_date, maturity_date, term_months, freq,
        principal, rate, outstanding_principal, outstanding_interest, total_outstanding, amount_repaid,
        n_installments, missed, dpd, classification, prov_rate, provision_amount, collateral, collateral_value,
        ltv, credit_score, kyc, restructured, write_off, write_off_amount, last_review, approving_authority,
        insurance, guarantor, emp_status, monthly_income, dti, source_sys
    ])

columns = [
    "Loan_ID", "Customer_ID", "Customer_Name", "Customer_Type", "Gender", "Age", "Sector",
    "Province", "District", "Branch_Code", "Branch_Name", "Loan_Officer", "Product_Type", "Currency",
    "Application_Date", "Approval_Date", "Disbursement_Date", "Maturity_Date", "Loan_Term_Months",
    "Repayment_Frequency", "Principal_Amount", "Interest_Rate_Percent", "Outstanding_Principal",
    "Outstanding_Interest", "Total_Outstanding", "Amount_Repaid_To_Date", "Number_of_Installments",
    "Missed_Payments_Count", "Days_Past_Due", "Loan_Classification", "Provisioning_Rate_Percent",
    "Provision_Amount", "Collateral_Type", "Collateral_Value", "Loan_To_Value_Ratio_Percent",
    "Credit_Score", "KYC_Status", "Restructured_Flag", "Write_Off_Flag", "Write_Off_Amount",
    "Last_Review_Date", "Approving_Authority", "Insurance_Flag", "Guarantor_Name", "Employment_Status",
    "Monthly_Income_RWF", "Debt_To_Income_Ratio_Percent", "Source_System"
]

df = pd.DataFrame(rows, columns=columns)
print(df.shape)
df.tail()

(50000, 48)


,Loan_ID,Customer_ID,Customer_Name,Customer_Type,Gender,Age,Sector,Province,District,Branch_Code,...,Write_Off_Flag,Write_Off_Amount,Last_Review_Date,Approving_Authority,Insurance_Flag,Guarantor_Name,Employment_Status,Monthly_Income_RWF,Debt_To_Income_Ratio_Percent,Source_System
49995,LN2019049995,CUS115503,Pascasie Uwizeyimana,Individual,Female,41.0,Education,Southern Province,Ruhango,BR004,...,No,0.0,2023-04-03,Branch Credit Committee,Yes,None,Unemployed,2883103.0,10.9,CoreBanking-T24
49996,LN2019049996,CUS102898,Josiane Kalisa,Individual,Female,34.0,Financial Services,Southern Province,Ruhango,BR002,...,No,0.0,2026-04-13,Board Credit Committee,No,Josiane Niyonsenga,Business Owner,2240827.0,46.8,LoanOriginationSystem
49997,LN2019049997,CUS103140,Umuganda Holdings,SME,N/A,NaN,Construction & Real Estate,Southern Province,Nyaruguru,BR003,...,No,0.0,2026-05-10,Head Office Credit Committee,No,Justin Gasana,N/A,NaN,NaN,CoreBanking-Flexcube
49998,LN2019049998,CUS107688,Justin Habimana,Individual,Male,35.0,Mining & Quarrying,Kigali City,Kicukiro,BR013,...,No,0.0,NaT,Branch Credit Committee,Yes,None,Retired,1672393.0,5.4,LoanOriginationSystem
49999,LN2019049999,CUS101859,Isange Enterprises,SME,N/A,NaN,Construction & Real Estate,Northern Province,Rulindo,BR001,...,No,0.0,2025-07-10,Regional Credit Committee,No,None,N/A,NaN,NaN,LegacyMigration2019


In [4]:
df.to_csv("loan_data.csv", index=False)

In [5]:
from google.colab import files
files.download("loan_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Power BI Exercises for Loan Data Analysis (Auditor's Perspective)

As a professional auditor, these exercises are designed to help you leverage Power BI for insightful analysis of the loan portfolio, focusing on risk, compliance, and financial performance. Assume you have already loaded the `loan_data.csv` into Power BI.

### **Part 1: Data Understanding & Initial Setup**

1.  **Data Loading and Inspection:**
    *   Load the `loan_data.csv` into Power BI Desktop.
    *   Review the data types of each column. Identify and correct any incorrect data types (e.g., dates stored as text, numeric values as text).
    *   Identify any columns with a high percentage of missing values. Discuss potential implications for analysis.
2.  **Basic Data Transformation:**
    *   Create a new column `Loan_Age_Days` (current date - `Disbursement_Date`).
    *   Create a new column `Remaining_Term_Months` (`Maturity_Date` - current date, converted to months). Handle past maturity dates appropriately.
    *   Group `Principal_Amount` into logical bins (e.g., 'Small', 'Medium', 'Large') to analyze loan size distribution.

### **Part 2: Key Performance Indicators & Risk Assessment**

1.  **Portfolio Overview Dashboard:**
    *   Design a dashboard that presents key aggregated metrics: Total Principal Amount, Total Outstanding Principal, Total Provision Amount, Number of Active Loans, Average Interest Rate.
    *   Use card visuals for these KPIs.
2.  **Delinquency Analysis:**
    *   Visualize the distribution of `Days_Past_Due` (DPD) using a histogram.
    *   Create a drill-down report showing `Total_Outstanding` by `Loan_Classification` and `Province`. Identify which classifications and regions contribute most to higher DPD.
    *   Calculate the percentage of `Outstanding_Principal` that is classified as 'Substandard', 'Doubtful', or 'Loss'.
3.  **Provisioning Adequacy:**
    *   Create a matrix visual showing `Provision_Amount` by `Loan_Classification` and `Branch_Name`.
    *   Calculate a measure for `Actual_Provisioning_Ratio` (`Provision_Amount` / `Total_Outstanding`) and compare it against the `Provisioning_Rate_Percent` to identify discrepancies.
4.  **Collateral Coverage:**
    *   For secured loans (where `Collateral_Type` is not 'None (Unsecured)'), calculate the `Average_LTV_Ratio_Percent` by `Product_Type`.
    *   Identify loans with an `LTV_Ratio_Percent` exceeding a certain threshold (e.g., 100%) to flag potentially under-collateralized loans.

### **Part 3: Compliance & Operational Efficiency**

1.  **KYC Status Monitoring:**
    *   Create a pie chart showing the distribution of `KYC_Status`.
    *   Filter to quickly identify customers with 'Incomplete' or 'Expired - Needs Update' KYC statuses. Create a table visual listing these `Customer_Name` and their `Loan_Officer`.
2.  **Loan Officer Performance:**
    *   Develop a report to compare `Loan_Officer` performance based on metrics like `Total_Principal_Amount` disbursed, `Average_Interest_Rate`, and `Average_Days_Past_Due` for their portfolio.
    *   Implement conditional formatting to highlight officers with higher-than-average DPD in their portfolio.
3.  **Data Source Reconciliation:**
    *   Analyze the `Source_System` column. Discuss how an auditor would use this information to ensure data completeness and consistency across different source systems.
    *   (Advanced) If possible, identify if any `Loan_ID` appears in multiple source systems unexpectedly (if duplicates were present, in this simulated data, they aren't, but it's a good thought exercise).

### **Part 4: Advanced Analysis & Dashboards**

1.  **Restructured and Write-Off Analysis:**
    *   Create a visual showing the trend of `Restructured_Flag` and `Write_Off_Flag` over time (using `Disbursement_Date`).
    *   Calculate the total `Write_Off_Amount` by `Sector` and `Approving_Authority`.
2.  **Customer Segmentation:**
    *   Segment customers by `Customer_Type`, `Age` groups, and `Monthly_Income_RWF` (if applicable) to understand loan behavior across different segments.
    *   Visualize the average `Credit_Score` for each `Customer_Type`.
3.  **Interactive Loan Portfolio Dashboard:**
    *   Consolidate relevant visuals into a comprehensive dashboard allowing filtering by `Province`, `Branch_Name`, `Product_Type`, and `Loan_Classification`.
    *   Ensure the dashboard includes clear indications of risk levels, portfolio size, and key trends.

## Audit Supervisor's Directives: Initial Loan Portfolio Review

*(To the Auditor, assuming no prior familiarity with this specific dataset)*

Good morning. I've been given access to this loan portfolio dataset, but I haven't had a chance to familiarize myself with its structure or content. Your task is to perform an initial review, focusing on areas that would typically raise red flags or require further investigation from an audit standpoint. Please prepare to discuss your findings on these points:

### **1. Portfolio Composition & Overview:**

*   **Size and Scope:** Can you give me a high-level summary of the total number of loans, the aggregate principal amount, and the total outstanding balances across the portfolio? What are the primary currencies involved, and what's their distribution?
*   **Product & Customer Mix:** What types of loan products are most prevalent? Are there particular customer segments (e.g., individual, SME, corporate) that dominate the portfolio by volume or value?
*   **Geographic Distribution:** Where are these loans geographically concentrated (provinces, districts)? Are there any surprising concentrations or dispersions we should be aware of?

### **2. Credit Risk Assessment:**

*   **Loan Classification Profile:** What is the overall distribution of loan classifications (Normal, Watch, Substandard, Doubtful, Loss)? Are there any significant shifts towards higher-risk categories?
*   **Delinquency Trends:** What are the average and maximum `Days_Past_Due` (DPD) across the portfolio? Can you identify any products, branches, or customer types exhibiting unusually high DPD figures?
*   **Provisioning Adequacy:** Based on the data, how does the calculated `Provision_Amount` relate to the `Provisioning_Rate_Percent` for each loan classification? Are there any significant discrepancies or under-provisioned segments?
*   **Collateral Coverage:** For loans with collateral, what is the general `Loan_To_Value_Ratio_Percent` (LTV)? Are there many instances where the LTV suggests insufficient collateral given the `Principal_Amount`? Specifically, identify any large loans where collateral appears to be weak or non-existent (excluding product types where collateral isn't expected).

### **3. Operational & Compliance Concerns:**

*   **KYC Status:** What is the state of our customer KYC records? Are there many customers flagged as 'Incomplete' or 'Expired - Needs Update'? What proportion of the portfolio value is associated with such statuses?
*   **Restructuring and Write-Offs:** What is the volume and value of restructured loans and loans that have been written off? Are these clustered around specific `Product_Type`s, `Sector`s, or `Approving_Authority`s?
*   **Loan Officer Performance:** Without judgment, can we observe any significant variations in the credit quality (e.g., average DPD, proportion of higher-risk classifications) of loans managed by different `Loan_Officer`s?
*   **Data Source Discrepancies:** Are there any anomalies or concentrations from specific `Source_System`s that might suggest data quality issues or unusual processing patterns?

### **4. Data Integrity & Anomalies:**

*   **Basic Sanity Checks:** Are there any loans where the `Outstanding_Principal` exceeds the `Principal_Amount`? Or where `Maturity_Date` precedes `Disbursement_Date`? Identify any other logical inconsistencies you come across.
*   **Missing Information:** Which fields have a notable amount of missing data? How might this impact our audit conclusions?

I need you to dig into these areas. Don't just give me numbers; try to identify patterns, potential risks, and areas where we might need to dig deeper. I'm looking for your audit intuition here.